# Recommendation systems

This notebook acompanies the slides _'Recommender Systems'_

#### Goal
Demonstrate how recommender systems work, how they can be developed and how they work internally.

This example assumes that we have a database of **movies** where some movies got **rated by users**.

Import all required libraries

In [1]:
from math import sqrt

Let's prepare some data.<br>
Our database consists of three attributes:<br>
- user
    - movie
    - user rating
We will use a dictionary to store the information.<br>
Here is the code

In [2]:
from recommendations import critics
critics

{'Lisa Rose': {'Lady in the Water': 2.5,
  'Snakes on a Plane': 3.5,
  'Just My Luck': 3.0,
  'Superman Returns': 3.5,
  'You, Me and Dupree': 2.5,
  'The Night Listener': 3.0},
 'Gene Seymour': {'Lady in the Water': 3.0,
  'Snakes on a Plane': 3.5,
  'Just My Luck': 1.5,
  'Superman Returns': 5.0,
  'The Night Listener': 3.0,
  'You, Me and Dupree': 3.5},
 'Michael Phillips': {'Lady in the Water': 2.5,
  'Snakes on a Plane': 3.0,
  'Superman Returns': 3.5,
  'The Night Listener': 4.0},
 'Claudia Puig': {'Snakes on a Plane': 3.5,
  'Just My Luck': 3.0,
  'The Night Listener': 4.5,
  'Superman Returns': 4.0,
  'You, Me and Dupree': 2.5},
 'Mick LaSalle': {'Lady in the Water': 3.0,
  'Snakes on a Plane': 4.0,
  'Just My Luck': 2.0,
  'Superman Returns': 3.0,
  'The Night Listener': 3.0,
  'You, Me and Dupree': 2.0},
 'Jack Matthews': {'Lady in the Water': 3.0,
  'Snakes on a Plane': 4.0,
  'The Night Listener': 3.0,
  'Superman Returns': 5.0,
  'You, Me and Dupree': 3.5},
 'Toby': {'Snak

We have a total of 7 users rating 6 movies in the range of 0..5

##### Distance function
The distance between two users is the difference of their ratings of films they have in common.

We want to compare user _Toby_ with user _Mick LaSalle_.

If Toby and LaSalle had just _'Superman Returns'_ in common their distance would be:

$$d = \sqrt {(4.0 - 3.0)^2 }$$

In [3]:
d = sqrt(pow((4.0-3.0),2))
d

1.0

But Toby and LaSalle have three films in common

In [4]:
[movie for movie in critics['Toby'] if movie in critics['Mick LaSalle']]

['Snakes on a Plane', 'You, Me and Dupree', 'Superman Returns']

We can thus calculate the distance as the squareroot of the sum of squares of difference of all the films

$~~~~~ d = \sqrt { \sum_i {(x_i - \overline {x})^2} }$


$\Rightarrow d = \sqrt { (4.5 - 4.0)^2 + (1.0 - 2.0)^2 + (4.0 - 3.0)^2 }$

In [5]:
d = sqrt(sum([pow((4.5-4.0),2), pow((1.0-2.0),2), pow((4.0-3.0),2)]))
d

1.5

Similarity should return a value between [0,1] with 0 indicating no match and 1 a match of 100%.

$~~~~~ d_{[0;1]} = \frac {1} {1 + \sqrt {\sum_i { (x_i - \overline {x})^2 } } }$

In [6]:
1 / (1 + d)

0.4

#### Similarity based on euclidian distance

In [7]:
from recommendations import similarity_distance
similarity_distance??

<br>
Test if we get the expected results

In [8]:
similarity_distance(critics, 'Toby', 'Mick LaSalle')

0.4

Now let's try _'Lisa Rose'_ and _'Claudia Puig'_

In [9]:
similarity_distance(critics, 'Lisa Rose', 'Claudia Puig')

0.38742588672279304

In [10]:
print('Lisa saw', critics['Lisa Rose'])
print('Claudia viewed', critics['Claudia Puig'])

Lisa saw {'Lady in the Water': 2.5, 'Snakes on a Plane': 3.5, 'Just My Luck': 3.0, 'Superman Returns': 3.5, 'You, Me and Dupree': 2.5, 'The Night Listener': 3.0}
Claudia viewed {'Snakes on a Plane': 3.5, 'Just My Luck': 3.0, 'The Night Listener': 4.5, 'Superman Returns': 4.0, 'You, Me and Dupree': 2.5}


In [11]:
print('Lisa Rose saw {0} that Claudia did not see'.format(
    [movie for movie in critics['Lisa Rose'] if movie not in critics['Claudia Puig']]))
print('Claudia Puig saw {0} that Lisa did not see'.format(
    [movie for movie in critics['Claudia Puig'] if movie not in critics['Lisa Rose']]))
print('They have these films in common:\n{0}'.format(
    [movie for movie in critics['Lisa Rose'] if movie in critics['Claudia Puig']]))

Lisa Rose saw ['Lady in the Water'] that Claudia did not see
Claudia Puig saw [] that Lisa did not see
They have these films in common:
['Snakes on a Plane', 'Just My Luck', 'Superman Returns', 'You, Me and Dupree', 'The Night Listener']


<br>
Assume that a person has <b>seen the same films as Toby</b>.<br>
However, this new user does not rate as enthusiastic as Toby. <br>Let's call this new user <i>'Wolf'</i> and create a set of ratings for him.

In [12]:
critics['Toby']

{'Snakes on a Plane': 4.5, 'You, Me and Dupree': 1.0, 'Superman Returns': 4.0}

In [13]:
critics['Wolf'] = {'Snakes on a Plane': 2.25, 'You, Me and Dupree': 0.5, 'Superman Returns': 2.0}

<br>
How does Wolf fare? (Intuition tells us that they should be pretty equal)

In [14]:
similarity_distance(critics, 'Toby', 'Wolf')

0.24681370272883318

<p style='font-size: 150%; color: #c00000;'>What?</p>

Euclidian distance distorts results if the data is not normalised (see feature normalisation).<br><br>
<font color='#c00000'>back to the slides</color><br><br>

#### Similarity based on Pearson correlation

In [15]:
from recommendations import similarity_pearson
similarity_pearson??

<br>
Test if we get the expected results

In [16]:
similarity_pearson(critics, 'Toby', 'Mick LaSalle')

0.9244734516419049

Remarkably better than the 0.4 with euclidian distance

And how does Wolf do?

In [17]:
similarity_pearson(critics, 'Toby', 'Wolf')

1.0

Pearson eliminates rating biases!

## Rate critics

We would like to find all users that correlate with a selected user in our dataset

In [18]:
from recommendations import topMatches
topMatches??

<br>
Let's test it. Give me the top three users who likes movies similarly to <i>Toby</i>

In [19]:
topMatches(critics, 'Toby', n=3)

[(1.0, 'Wolf'),
 (0.9912407071619299, 'Lisa Rose'),
 (0.9244734516419049, 'Mick LaSalle')]

<br><br>
<font color='#c00000'>back to the slides</color><br><br>

## Calculate recommendations

In [20]:
from recommendations import getRecommendations
getRecommendations??

This function takes the comparison metric as an optional parameter (defaults to Pearson). It is O(2k).

We could now try to find recommendations for user _'Toby'_

In [21]:
getRecommendations(critics, 'Toby')

[(3.3477895267131017, 'The Night Listener'),
 (2.8325499182641614, 'Lady in the Water'),
 (2.530980703765565, 'Just My Luck')]

<br>
It seems that our best bet would be to watch <i>'The Night Listener'</i> (Doesn't sound to enthusiastic though).<br>
N.b. Euclidian distance returns similar recommendations.<br>

In [22]:
# clean up data
del critics['Wolf']

## Find similar preferences

If items (movies) are our primary target, we need to transpose the database to make items the primary key. This needs to be done irregularly (when your dataset has changed significantly).

In [23]:
from recommendations import transformPrefs
transformPrefs??

In [24]:
movies = transformPrefs(critics)
movies

{'Lady in the Water': {'Lisa Rose': 2.5,
  'Gene Seymour': 3.0,
  'Michael Phillips': 2.5,
  'Mick LaSalle': 3.0,
  'Jack Matthews': 3.0},
 'Snakes on a Plane': {'Lisa Rose': 3.5,
  'Gene Seymour': 3.5,
  'Michael Phillips': 3.0,
  'Claudia Puig': 3.5,
  'Mick LaSalle': 4.0,
  'Jack Matthews': 4.0,
  'Toby': 4.5},
 'Just My Luck': {'Lisa Rose': 3.0,
  'Gene Seymour': 1.5,
  'Claudia Puig': 3.0,
  'Mick LaSalle': 2.0},
 'Superman Returns': {'Lisa Rose': 3.5,
  'Gene Seymour': 5.0,
  'Michael Phillips': 3.5,
  'Claudia Puig': 4.0,
  'Mick LaSalle': 3.0,
  'Jack Matthews': 5.0,
  'Toby': 4.0},
 'You, Me and Dupree': {'Lisa Rose': 2.5,
  'Gene Seymour': 3.5,
  'Claudia Puig': 2.5,
  'Mick LaSalle': 2.0,
  'Jack Matthews': 3.5,
  'Toby': 1.0},
 'The Night Listener': {'Lisa Rose': 3.0,
  'Gene Seymour': 3.0,
  'Michael Phillips': 4.0,
  'Claudia Puig': 4.5,
  'Mick LaSalle': 3.0,
  'Jack Matthews': 3.0}}

<br>
Now let's find movies that are similarly popular to <i>'Superman Returns'</i>.

In [25]:
topMatches(movies, 'Superman Returns')

[(0.6579516949597695, 'You, Me and Dupree'),
 (0.4879500364742689, 'Lady in the Water'),
 (0.11180339887498941, 'Snakes on a Plane'),
 (-0.1798471947990544, 'The Night Listener'),
 (-0.42289003161103106, 'Just My Luck')]

<br>
And who could possibly be interested in watching these movies (and haven't seen it so far)?

In [26]:
getRecommendations(movies, 'You, Me and Dupree')

[(3.1637361366111816, 'Michael Phillips')]

In [27]:
getRecommendations(movies, 'Just My Luck')

[(4.0, 'Michael Phillips'), (3.0, 'Jack Matthews')]

In [28]:
getRecommendations(movies, 'Lady in the Water')

[(3.610031066802182, 'Toby'), (3.4436241497684494, 'Claudia Puig')]

<br>
We might conclude that Michael would not like <i>'You, Me and Dupree'</i> but might be interested to see <i>'Just My Luck'</i>.<br>
Claudia on the other hand might want to see <i>'Lady in the Water'</i>.<br>

<br><br>
<font color='#c00000'>back to the slides</color><br><br>

## Find similar items

We find similar items by calculating the similarity score with all other items

In [29]:
from recommendations import calculateSimilarItems
calculateSimilarItems??

In [31]:
similar_items = calculateSimilarItems(critics, similarity=similarity_distance)
similar_items

{'Lady in the Water': [(0.4494897427831781, 'You, Me and Dupree'),
  (0.38742588672279304, 'The Night Listener'),
  (0.3483314773547883, 'Snakes on a Plane'),
  (0.3483314773547883, 'Just My Luck'),
  (0.2402530733520421, 'Superman Returns')],
 'Snakes on a Plane': [(0.3483314773547883, 'Lady in the Water'),
  (0.32037724101704074, 'The Night Listener'),
  (0.3090169943749474, 'Superman Returns'),
  (0.2553967929896867, 'Just My Luck'),
  (0.1886378647726465, 'You, Me and Dupree')],
 'Just My Luck': [(0.3483314773547883, 'Lady in the Water'),
  (0.32037724101704074, 'You, Me and Dupree'),
  (0.2989350844248255, 'The Night Listener'),
  (0.2553967929896867, 'Snakes on a Plane'),
  (0.20799159651347807, 'Superman Returns')],
 'Superman Returns': [(0.3090169943749474, 'Snakes on a Plane'),
  (0.252650308587072, 'The Night Listener'),
  (0.2402530733520421, 'Lady in the Water'),
  (0.20799159651347807, 'Just My Luck'),
  (0.1918253663634734, 'You, Me and Dupree')],
 'You, Me and Dupree': [

<br>
Similarity calculation is based on user preferences!<br>
Using Pearson metric to calculate similarities returns slightly modified rankings.<br>
If feature based similarity is required, items need to be rated according to a weighted set of features.<br>

## Get Recommendations based on user preferences and item similarities

In [33]:
from recommendations import getRecommendedItems
getRecommendedItems??

In [34]:
getRecommendedItems(critics, similar_items, 'Toby')

[(3.1667425234070894, 'The Night Listener'),
 (2.9366294028444346, 'Just My Luck'),
 (2.868767392626467, 'Lady in the Water')]

Remember

In [35]:
getRecommendations(critics, 'Toby')

[(3.3477895267131017, 'The Night Listener'),
 (2.8325499182641614, 'Lady in the Water'),
 (2.530980703765565, 'Just My Luck')]

<br>
Both algorithms recommend <i>'The Night Listener'</i>. Second and third place ranking has changed a little.<br><br>
Recap that using element based filtering does not require on-the-fly calculations and can be prepared in advance. It requires recomputation only when the datasets change significantly. Collaborative filtering (as carried out in <i>getRecommendations</i>) is computing intensive.<br><br>
If datasets are sparsely populated, element based filtering might lead to distorted results (we compare items based on user preferences - there needs to be some overlap of ratings for the algorith to work effectively).

<br><br>
<font color='#c00000'>End of notebook<br>back to the slides</color><br><br>

<br>
<h1>Examples with students data</h1>
<br>

In [ ]:
from recommendations import read_csv
films = read_csv('films.csv')
# these are some lines
[name for name in films.keys()]

In [ ]:
similarity_pearson(films, 'olka_dorn', 'Manuel Plech')

In [ ]:
topMatches(films, 'Carlos Rodriguez', n=5)

In [ ]:
getRecommendations(films, 'Ale')

In [ ]:
similar_films = calculateSimilarItems(films, similarity=similarity_distance)
getRecommendedItems(films, similar_films, 'Matthias Rosskopf', n=5)

<h4><font color="green">Exercises: </font></h4>
<ol>
    <li>Read about the Jaccard Score and implement it. When is it suitable to use?</li>
    <li>Exchange ratings with tags (e.g. genre). Can you still find similarities?</li>
    <li>Rewrite user-based (collaborative) filtering to pre-compute the similarity matrix.<br>
        A function body has been added to the source file.</li>
    <li>Apply to different datasets (create your own)</li>
    <li>Rewrite using NumPy for computation. Is this implementation faster?<br>
        (Use Timeit to compare runtime performance)</li>
</ol>